# G9 雙塔架構（DualTower）Fine-tuning

本 notebook 實作 G9 Siamese 雙塔分類器：title 與 content 各自通過獨立 tokenize（共享 XLM-R backbone），
再以 SBERT 式合併向量 `[h_t; h_c; |h_t−h_c|; h_t⊙h_c]` 送入分類頭。

與 G8 唯一差異在架構：content 有獨立的 256 token budget，不與 title 共搶預算。

## 環境安裝

In [ ]:
# 在 Colab 上安裝所需套件；本機執行時可略過。
!pip install -q transformers datasets accelerate scikit-learn

## 掛載 Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 路徑與超參數設定

In [ ]:
from pathlib import Path

DRIVE_PROJECT    = Path("/content/drive/MyDrive/NLP_Project")
PROCESSED_DIR    = DRIVE_PROJECT / "dataset" / "processed"

MODEL_NAME    = "xlm-roberta-base"
BATCH_SIZE    = 16
NUM_EPOCHS    = 3
LEARNING_RATE = 2e-5
SEED          = 42

# G9 雙塔專屬設定
TITLE_MAX_LEN      = 64
CONTENT_MAX_LEN    = 256
RUN_TAG            = "g9"
USE_ADVERSARIAL_G8 = True   # G9 沿用 G8 訓練資料（隔離架構變因）
THRESHOLD = 0.50

TOKEN_DROPOUT_PROB = 0.1
CONTENT_CROP_PROB  = 0.5

MODEL_OUTPUT_DIR = DRIVE_PROJECT / f"models/xlm-roberta-clickbait-{RUN_TAG}-dualtower"
METRICS_OUT      = DRIVE_PROJECT / f"results/{RUN_TAG}_transformer_metrics.json"

print(f"Run tag: {RUN_TAG}")
print(f"Model output: {MODEL_OUTPUT_DIR}")
print(f"Metrics output: {METRICS_OUT}")
print(f"Title max len: {TITLE_MAX_LEN} | Content max len: {CONTENT_MAX_LEN}")

## 匯入套件

In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModel,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## 載入資料

In [ ]:
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
valid_df = pd.read_csv(PROCESSED_DIR / "valid.csv")
test_df  = pd.read_csv(PROCESSED_DIR / "test.csv")

# G9 沿用 G8 對抗資料，與 G8 訓練資料相同（隔離架構變因）
if USE_ADVERSARIAL_G8:
    adv_path = PROCESSED_DIR / "adversarial_g8.csv"
    adv = pd.read_csv(adv_path)
    train_df = pd.concat([train_df, adv], ignore_index=True)
    print(f"G9 (USE_ADVERSARIAL_G8=True): 併入對抗資料 {len(adv)} 筆，總訓練筆數 {len(train_df)}")

for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    assert set(df.columns) >= {"id", "lang", "title", "content", "label"}, f"{name} missing columns"
    assert set(df["label"].unique()).issubset({0, 1}), f"{name} has unexpected labels"
    assert {"zh", "en"}.issubset(set(df["lang"].unique())), f"{name} missing a language"
    print(f"{name}: {len(df):,} rows | label 0={df['label'].eq(0).sum():,} label 1={df['label'].eq(1).sum():,}")

In [ ]:
# ── 【可選】刪減 tw_neg_only，測 Bug3 已馴服後能否再降 FN ──
# 放在「載入資料」cell 之後執行。改 KEEP_RATIO 一個數即可切 A/B/C/D。
# 1.0=全留(A) / 0.5=B / 0.25=C / 0.0=全刪(D)。不動任何 csv。
TW_NEG_KEEP_RATIO = 0.5

if USE_ADVERSARIAL_G8 and TW_NEG_KEEP_RATIO < 1.0:
    is_twneg = train_df["id"].astype(str).str.endswith("_twneg")
    twneg = train_df[is_twneg]
    keep_n = int(round(len(twneg) * TW_NEG_KEEP_RATIO))
    twneg_keep = twneg.sample(n=keep_n, random_state=SEED) if keep_n > 0 else twneg.iloc[0:0]
    train_df = pd.concat([train_df[~is_twneg], twneg_keep], ignore_index=True)
    print(f"tw_neg_only: {int(is_twneg.sum())} → 保留 {keep_n} (ratio={TW_NEG_KEEP_RATIO})")
    print(f"訓練筆數: {len(train_df)} | label 0={train_df['label'].eq(0).sum()} label 1={train_df['label'].eq(1).sum()}")
else:
    print(f"tw_neg_only 未刪減 (G8={USE_ADVERSARIAL_G8}, ratio={TW_NEG_KEEP_RATIO})")


## DualTowerDataset 與資料增強

G9 的核心差異：title 與 content 分別 tokenize，各有獨立 token budget（64 + 256）。
增強邏輯（token dropout + content crop）與 G8 相同，確保只有架構不同。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
MASK_TOKEN_ID = tokenizer.mask_token_id


def augment_content(content, crop_prob):
    """有 crop_prob 的機率從隨機位置截取 content，讓模型見到內文不同片段"""
    if not content or random.random() > crop_prob:
        return content
    words = content.split()
    if len(words) <= 1:
        return content
    start = random.randint(0, len(words) // 2)
    return " ".join(words[start:])


def apply_token_dropout(input_ids, attention_mask, mask_id, dropout_prob):
    """把非特殊 token 以 dropout_prob 的機率換成 <mask>，增強模型泛化"""
    special_ids = {
        tokenizer.cls_token_id, tokenizer.sep_token_id,
        tokenizer.pad_token_id, tokenizer.eos_token_id,
    }
    mask = (
        (torch.rand_like(input_ids.float()) < dropout_prob)
        & attention_mask.bool()
        & ~torch.isin(input_ids, torch.tensor(list(special_ids)))
    )
    input_ids[mask] = mask_id
    return input_ids


class DualTowerDataset(Dataset):
    """G9 雙塔 Dataset：title 與 content 分別 tokenize，各有獨立 token budget"""

    def __init__(self, df, tokenizer, augment=False):
        self.labels   = df["label"].tolist()
        self.titles   = df["title"].fillna("").tolist()
        self.contents = df["content"].fillna("").tolist()
        self.tokenizer = tokenizer
        self.augment   = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        title   = self.titles[idx]
        content = self.contents[idx]

        if self.augment:
            content = augment_content(content, CONTENT_CROP_PROB)

        enc_t = self.tokenizer(
            title,
            max_length=TITLE_MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        enc_c = self.tokenizer(
            content,
            max_length=CONTENT_MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

        t_ids, t_mask = enc_t["input_ids"].squeeze(0), enc_t["attention_mask"].squeeze(0)
        c_ids, c_mask = enc_c["input_ids"].squeeze(0), enc_c["attention_mask"].squeeze(0)

        if self.augment:
            t_ids = apply_token_dropout(t_ids.clone(), t_mask, MASK_TOKEN_ID, TOKEN_DROPOUT_PROB)
            c_ids = apply_token_dropout(c_ids.clone(), c_mask, MASK_TOKEN_ID, TOKEN_DROPOUT_PROB)

        return {
            "title_input_ids":        t_ids,
            "title_attention_mask":   t_mask,
            "content_input_ids":      c_ids,
            "content_attention_mask": c_mask,
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


train_dataset = DualTowerDataset(train_df, tokenizer, augment=True)
valid_dataset = DualTowerDataset(valid_df, tokenizer, augment=False)
test_dataset  = DualTowerDataset(test_df,  tokenizer, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=True)

print(f"train batches: {len(train_loader)} | valid batches: {len(valid_loader)} | test batches: {len(test_loader)}")

## 資料集基本測試

In [ ]:
sample_batch = next(iter(train_loader))
assert sample_batch["title_input_ids"].shape   == (BATCH_SIZE, TITLE_MAX_LEN),   f"title shape error"
assert sample_batch["title_attention_mask"].shape == (BATCH_SIZE, TITLE_MAX_LEN), f"title mask shape error"
assert sample_batch["content_input_ids"].shape == (BATCH_SIZE, CONTENT_MAX_LEN), f"content shape error"
assert sample_batch["content_attention_mask"].shape == (BATCH_SIZE, CONTENT_MAX_LEN), f"content mask shape error"
assert sample_batch["labels"].max().item() <= 1
assert sample_batch["labels"].min().item() >= 0
print("DualTowerDataset smoke test passed.")
print("title_input_ids shape:",   sample_batch["title_input_ids"].shape)
print("content_input_ids shape:", sample_batch["content_input_ids"].shape)

## DualTowerClassifier 模型定義

Siamese 架構（共享 XLM-R backbone），合併方式：`[h_t; h_c; |h_t−h_c|; h_t⊙h_c]` → 3072 維。
分類頭：`Linear(3072→768) → Tanh → Dropout(0.1) → Linear(768→2)`。

In [ ]:
class DualTowerClassifier(nn.Module):
    def __init__(self, model_name, hidden_size=768, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 4, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def mean_pool(self, last_hidden, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        return (last_hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

    def encode(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        return self.mean_pool(out.last_hidden_state, attention_mask)

    def forward(self, title_input_ids, title_attention_mask,
                content_input_ids, content_attention_mask, labels=None):
        h_t = self.encode(title_input_ids,   title_attention_mask)
        h_c = self.encode(content_input_ids, content_attention_mask)
        v   = torch.cat([h_t, h_c, (h_t - h_c).abs(), h_t * h_c], dim=-1)
        logits = self.classifier(v)
        loss = self.loss_fn(logits, labels) if labels is not None else None
        return logits, loss


model = DualTowerClassifier(MODEL_NAME)
model = model.to(DEVICE)

total_params    = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")

## 訓練迴圈

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"Total steps: {total_steps} | Warmup steps: {warmup_steps}")

In [ ]:
def evaluate(model, loader, device):
    """在給定 DataLoader 上跑推理，回傳 loss、accuracy 與 macro F1"""
    model.eval()
    all_preds, all_labels, total_loss = [], [], 0.0

    with torch.no_grad():
        for batch in loader:
            title_input_ids        = batch["title_input_ids"].to(device)
            title_attention_mask   = batch["title_attention_mask"].to(device)
            content_input_ids      = batch["content_input_ids"].to(device)
            content_attention_mask = batch["content_attention_mask"].to(device)
            labels                 = batch["labels"].to(device)

            logits, loss = model(
                title_input_ids=title_input_ids,
                title_attention_mask=title_attention_mask,
                content_input_ids=content_input_ids,
                content_attention_mask=content_attention_mask,
                labels=labels,
            )
            total_loss += loss.item()

            probs = torch.softmax(logits.logits, dim=-1)[:, 1]
            preds = (probs >= THRESHOLD).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    return avg_loss, acc, f1


MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_valid_f1 = 0.0
history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_loss = 0.0

    for step, batch in enumerate(train_loader, 1):
        title_input_ids        = batch["title_input_ids"].to(DEVICE)
        title_attention_mask   = batch["title_attention_mask"].to(DEVICE)
        content_input_ids      = batch["content_input_ids"].to(DEVICE)
        content_attention_mask = batch["content_attention_mask"].to(DEVICE)
        labels                 = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        logits, loss = model(
            title_input_ids=title_input_ids,
            title_attention_mask=title_attention_mask,
            content_input_ids=content_input_ids,
            content_attention_mask=content_attention_mask,
            labels=labels,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        if step % 500 == 0:
            current_lr = optimizer.param_groups[0]["lr"]
            print(f"  Epoch {epoch} step {step}/{len(train_loader)} | loss={train_loss/step:.4f} | lr={current_lr:.2e}")

    avg_train_loss = train_loss / len(train_loader)
    val_loss, val_acc, val_f1 = evaluate(model, valid_loader, DEVICE)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch} | train_loss={avg_train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | val_f1={val_f1:.4f} | lr={current_lr:.2e}")

    if val_f1 > best_valid_f1:
        best_valid_f1 = val_f1
        # DualTowerClassifier 不能直接 save_pretrained；分別存 backbone 與分類頭
        model.encoder.save_pretrained(MODEL_OUTPUT_DIR)
        tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
        torch.save(model.classifier.state_dict(),
                   MODEL_OUTPUT_DIR / "classifier_head.pt")
        print(f"  -> Best model saved (val_f1={val_f1:.4f})")

print(f"\nTraining done. Best val F1: {best_valid_f1:.4f}")

## 訓練曲線

In [ ]:
RESULTS_DIR = DRIVE_PROJECT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

epochs_range = range(1, NUM_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_range, history["train_loss"], marker="o", label="Train Loss")
ax1.plot(epochs_range, history["val_loss"],   marker="o", label="Val Loss")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Train / Val Loss (G9)"); ax1.legend(); ax1.grid(True)

ax2.plot(epochs_range, history["val_f1"],  marker="o", label="Val Macro F1", color="green")
ax2.plot(epochs_range, history["val_acc"], marker="o", label="Val Accuracy",  color="orange")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Score")
ax2.set_title("Val F1 & Accuracy (G9)"); ax2.legend(); ax2.grid(True)

fig.tight_layout()
curve_path = RESULTS_DIR / f"{RUN_TAG}_transformer_training_curve.png"
fig.savefig(curve_path, dpi=150)
plt.show()
print("Training curve saved to:", curve_path)

## 測試集評估

In [ ]:
# 重新載入最佳 checkpoint
best_model = DualTowerClassifier(MODEL_NAME)
best_model.encoder = AutoModel.from_pretrained(MODEL_OUTPUT_DIR)
best_model.classifier.load_state_dict(
    torch.load(MODEL_OUTPUT_DIR / "classifier_head.pt", map_location=DEVICE)
)
best_model = best_model.to(DEVICE)
best_model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader:
        logits, _ = best_model(
            title_input_ids=batch["title_input_ids"].to(DEVICE),
            title_attention_mask=batch["title_attention_mask"].to(DEVICE),
            content_input_ids=batch["content_input_ids"].to(DEVICE),
            content_attention_mask=batch["content_attention_mask"].to(DEVICE),
        )
        probs = torch.softmax(logits.logits, dim=-1)[:, 1]
        preds = (probs >= THRESHOLD).long()
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["labels"].numpy())

print("=== Test Set Results (G9) ===")
print(classification_report(all_labels, all_preds, target_names=["non-clickbait", "clickbait"]))
print("Confusion matrix:")
print(confusion_matrix(all_labels, all_preds))

## 儲存評估結果

In [ ]:
RESULTS_DIR = DRIVE_PROJECT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Classification report → CSV ---
report_dict = classification_report(
    all_labels, all_preds,
    target_names=["non-clickbait", "clickbait"],
    output_dict=True,
)
report_df = pd.DataFrame(report_dict).transpose()
report_path = RESULTS_DIR / f"{RUN_TAG}_transformer_metrics.csv"
report_df.to_csv(report_path, encoding="utf-8-sig")
print("Metrics saved to:", report_path)
print(report_df.round(4))

# --- Metrics JSON (accuracy + macro_f1) ---
metrics_json = {
    "run_tag": RUN_TAG,
    "accuracy": report_dict["accuracy"],
    "macro_f1": report_dict["macro avg"]["f1-score"],
}
with open(METRICS_OUT, "w") as f:
    json.dump(metrics_json, f, indent=2)
print("JSON metrics saved to:", METRICS_OUT)

# --- Confusion matrix → PNG ---
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_xticklabels(["non-clickbait", "clickbait"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["non-clickbait", "clickbait"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"XLM-RoBERTa Confusion Matrix ({RUN_TAG.upper()} DualTower)")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
fig.tight_layout()
cm_path = RESULTS_DIR / f"{RUN_TAG}_transformer_confusion_matrix.png"
fig.savefig(cm_path, dpi=150)
plt.show()
print("Confusion matrix saved to:", cm_path)

## Task：驗證 B 複刻（內文利用率測試）

固定標題、換不同內文，觀察 G8（單塔）vs G9（雙塔）的預測機率是否隨內文改變——
G9 的機率分散度若大於 G8，本身就是 content utilization 改善的直接證據。

**兩層分析**：

1. **Within-title group**（主要指標）
   固定同一個誘導標題（`TITLE_BC`），換 3 種內文：兌現 / 廣告 / 空。
   在此組內算 std，排除 title sensitivity 干擾。比較 G8 std vs G9 std。

2. **Cross-title cases**（原始 4 case，保留作對照）
   Case A 是 title 消融對比（不同標題），不是 within-title 測試；
   Case D 是 sanity check。
   此組 std 含 title sensitivity，不單獨作為 content utilization 的證據。

**方向判斷**：由 ground-truth label 決定，不套用「廣告/空內文 → p 下降」的錯誤假設。
誘導標題 + 廣告/空內文 = label 1（典型 clickbait），G9 若學對應維持或增強 p(clickbait)。

此 cell **不依賴訓練 loop**，可單獨執行：
- 需先執行測試集評估 cell 取得 `best_model`（G9）
- helpers cell 會自動載入 G8 checkpoint（`models/xlm-roberta-clickbait-g8`）以產生對照
- 本 cell 已自包含 G8 vs G9 std 比較，不再依賴外部 `explain_tokens.py`

In [ ]:
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification


def predict_g9(title, content, model, tokenizer, device):
    """回傳 G9 雙塔對 (title, content) 預測為 clickbait 的機率"""
    model.eval()
    with torch.no_grad():
        enc_t = tokenizer(
            title,
            max_length=TITLE_MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        enc_c = tokenizer(
            content,
            max_length=CONTENT_MAX_LEN,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        logits, _ = model(
            title_input_ids=enc_t["input_ids"].to(device),
            title_attention_mask=enc_t["attention_mask"].to(device),
            content_input_ids=enc_c["input_ids"].to(device),
            content_attention_mask=enc_c["attention_mask"].to(device),
        )
        return F.softmax(logits, dim=-1)[0, 1].item()


# ── G8 單塔對照（成功條件 2 要求 G8 vs G9 std 比較）──────────────
# G8 用 tokenizer(title, content) 拼接成單一序列，與 G9 雙塔分開 tokenize 不同。
G8_MODEL_DIR = DRIVE_PROJECT / "models" / "xlm-roberta-clickbait-g8"
G8_MAX_LENGTH = 256

g8_model = AutoModelForSequenceClassification.from_pretrained(G8_MODEL_DIR).to(DEVICE)
g8_model.eval()
g8_tokenizer = AutoTokenizer.from_pretrained(G8_MODEL_DIR)


def predict_g8(title, content, model=g8_model, tokenizer=g8_tokenizer, device=DEVICE):
    """回傳 G8 單塔對 (title, content) 預測為 clickbait 的機率（title </s> content 拼接）"""
    model.eval()
    with torch.no_grad():
        enc = tokenizer(
            title,
            content,
            max_length=G8_MAX_LENGTH,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        ).logits
        return F.softmax(logits, dim=-1)[0, 1].item()


print("predict_g9 and predict_g8 defined (G8 checkpoint loaded for comparison).")

In [ ]:
TITLE_BC = "探討優質睡眠與深層放鬆的生理機制：特定種類的植物精油成分有助於副交感神經的穩定"
CONTENT_FULFILLED = (
    "研究顯示，薰衣草中的沉香醇（linalool）可透過抑制興奮性神經傳遞物質、"
    "促進 GABA 受體活性，進而降低心跳速率並穩定副交感神經系統，改善入睡效率。"
    "實驗以雙盲設計追蹤120名受試者四週，薰衣草組平均入睡時間縮短14分鐘。"
)
# 真實廣告軟文段落，比「广告」兩字更能代表 Bug 4 的實際場景
CONTENT_AD = (
    "本產品採用天然植物精華配方，經過嚴格品質把關，讓您每天都能感受到煥然一新的活力。"
    "立即點擊下方連結，享受限時85折優惠，加入LINE好友即可獲得精美試用包。"
    "全館滿999免運，現在下單再享買二送一優惠，數量有限，售完為止！"
)
CONTENT_EMPTY = ""

# ── Part 1：Within-title group（固定 TITLE_BC，只換內文）─────────────
within_title_cases = [
    (CONTENT_FULFILLED, 0, "同誘導標題 + 兌現內文 → gt=0，p 應低"),
    (CONTENT_AD,        1, "同誘導標題 + 廣告軟文內文 → gt=1，p 應高（Bug 4 典型）"),
    (CONTENT_EMPTY,     1, "同誘導標題 + 空內文 → gt=1，p 應高"),
]

print("=" * 70)
print("【Part 1】Within-title group（固定標題，換內文）")
print(f"  Title: {TITLE_BC}")
print("=" * 70)

g8_wt, g9_wt = [], []
for content, gt, desc in within_title_cases:
    p8 = predict_g8(TITLE_BC, content)
    p9 = predict_g9(TITLE_BC, content, best_model, tokenizer, DEVICE)
    g8_wt.append(p8)
    g9_wt.append(p9)
    ok8 = (p8 > THRESHOLD) == bool(gt)
    ok9 = (p9 > THRESHOLD) == bool(gt)
    print(f"  [{desc}]")
    print(f"    G8 p={p8:.4f} {'✓' if ok8 else '✗'}   G9 p={p9:.4f} {'✓' if ok9 else '✗'}")

g8_wt_std = float(np.std(g8_wt))
g9_wt_std = float(np.std(g9_wt))
print(f"\n  G8 within-title std: {g8_wt_std:.4f}")
print(f"  G9 within-title std: {g9_wt_std:.4f}")
print(f"  （G9 std > G8 std 代表 G9 比 G8 更有感知到內文差異，即 content utilization 改善）")

# ── Part 2：Cross-title cases（4 case + 1 英文 case）────────────────
TITLE_PLAIN_ZH = "薰衣草精油可改善睡眠品質，研究：與副交感神經活化有關"
TITLE_INDUCED_EN = (
    "Examining the Neurological Basis of Deep Rest: "
    "A Specific Category of Plant-derived Compounds May Enhance Parasympathetic Activity"
)
CONTENT_AD_EN = (
    "Try our premium sleep supplement today! Clinically inspired formula. "
    "Click the link below for a limited-time 20% discount. "
    "Subscribe now and get a free trial pack with your first order."
)

cross_title_cases = [
    (
        "探討優質睡眠的生理機制：薰衣草精油成分有助於副交感神經穩定",
        CONTENT_FULFILLED,
        0,
        "Case A：title 消融對比（直白標題 + 兌現內文）→ gt=0，p 應低",
    ),
    (TITLE_BC, CONTENT_AD,   1, "Case B：誘導標題 + 廣告軟文 → gt=1，p 應高（Bug 4 典型）"),
    (TITLE_BC, CONTENT_EMPTY, 1, "Case C：誘導標題 + 空內文 → gt=1，p 應高"),
    (
        TITLE_PLAIN_ZH,
        "研究顯示，薰衣草中的沉香醇（linalool）可透過抑制興奮性神經傳遞物質來穩定副交感神經。",
        0,
        "Case D：sanity check（平實標題 + 兌現內文）→ gt=0，p 應低",
    ),
    (
        TITLE_INDUCED_EN,
        CONTENT_AD_EN,
        1,
        "Case E：英文誘導標題 + 英文廣告內文 → gt=1，p 應高（Bug 4 英文版）",
    ),
]

print("\n" + "=" * 70)
print("【Part 2】Cross-title cases（std 含 title sensitivity，參考用）")
print("=" * 70)

g8_ct, g9_ct = [], []
for title, content, gt, desc in cross_title_cases:
    p8 = predict_g8(title, content)
    p9 = predict_g9(title, content, best_model, tokenizer, DEVICE)
    g8_ct.append(p8)
    g9_ct.append(p9)
    ok8 = (p8 > 0.5) == bool(gt)
    ok9 = (p9 > 0.5) == bool(gt)
    print(f"  [{desc}]")
    print(f"    G8 p={p8:.4f} {'✓' if ok8 else '✗'}   G9 p={p9:.4f} {'✓' if ok9 else '✗'}")

g8_ct_std = float(np.std(g8_ct))
g9_ct_std = float(np.std(g9_ct))

print("\n" + "=" * 70)
print("[Summary] G8 vs G9 機率分散度（成功條件 2）")
print(f"  Within-title std（主要指標，排除 title sensitivity）:")
print(f"    G8 = {g8_wt_std:.4f}   G9 = {g9_wt_std:.4f}   "
      f"{'✓ G9 > G8（content utilization 改善）' if g9_wt_std > g8_wt_std else '✗ G9 未超過 G8'}")
print(f"  Cross-title std （含 title sensitivity，參考用）:")
print(f"    G8 = {g8_ct_std:.4f}   G9 = {g9_ct_std:.4f}")
print("=" * 70)
